LINEAR PREDICTION MODEL TO TRACK CAR PRICE

In [6]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder # for scaling and encoding
from sklearn.model_selection import train_test_split


initial = pd.read_csv('used_car_price_prediction_part1.csv')
# Split into 3 roughly equal parts (adjust number as needed)
chunks = 3
chunk_size = len(car_df) // chunks

for i in range(chunks):
    start = i * chunk_size
    end = None if i == chunks - 1 else (i + 1) * chunk_size
    car_df.iloc[start:end].to_csv(f"used_car_price_prediction_part{i+1}.csv", index=False)
    
car1_df = pd.read_csv('used_car_price_prediction_part1.csv')
car1_df

,Brand,Model,Year,Mileage_kmpl,Engine_CC,Horsepower,Fuel_Type,Transmission,Owner_Type,Color,City,Kms_Driven,Insurance_Valid,Service_History,Accidents,Tax_Paid,Number_of_Doors,Seats,Registration_Age,Price
0,Toyota,Camry,2001,19.615852,1396.560379,NaN,Hybrid,Automatic,First,Grey,Chennai,500928,1,1,0,1,4,5,24,3.531896e+05
1,Hyundai,i20,2012,19.478608,1130.771005,137.021719,Petrol,NaN,First,Grey,Mumbai,211510,0,1,0,1,4,5,13,9.596943e+05
2,Mahindra,Bolero,2008,17.469920,1766.466250,142.902961,NaN,Automatic,First,Black,Kolkata,333999,1,0,0,1,5,5,17,2.920229e+05
3,BMW,X1,2015,21.295421,NaN,49.782079,NaN,Manual,First,White,Pune,225930,1,1,3,1,4,5,10,2.949453e+06
4,Honda,Civic,2022,18.326833,1239.334935,121.505993,Diesel,Manual,Second,White,Kolkata,43404,1,0,0,1,5,5,3,6.365208e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111661,Tata,Punch,2005,NaN,1761.529435,139.389685,Hybrid,Manual,Fourth+,Black,Kolkata,432240,1,1,0,1,5,5,20,5.000000e+04
111662,Tata,Harrier,2014,22.871347,1156.299967,85.496967,Petrol,Automatic,First,Blue,Kolkata,263604,1,1,0,1,4,5,11,4.894001e+05
111663,Ford,EcoSport,2008,23.291805,1336.926351,96.292228,Diesel,Manual,First,Blue,Pune,2423316,1,1,0,1,4,7,17,3.558083e+06
111664,Ford,Figo,2015,11.734531,2431.715340,NaN,Petrol,Manual,Second,Blue,NaN,229370,1,1,0,1,5,5,10,7.322810e+05


In [7]:
car1_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 111666 entries, 0 to 111665
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Brand             111666 non-null  str    
 1   Model             111666 non-null  str    
 2   Year              111666 non-null  int64  
 3   Mileage_kmpl      102675 non-null  float64
 4   Engine_CC         102625 non-null  float64
 5   Horsepower        102763 non-null  float64
 6   Fuel_Type         103079 non-null  str    
 7   Transmission      102705 non-null  str    
 8   Owner_Type        111666 non-null  str    
 9   Color             102629 non-null  str    
 10  City              102795 non-null  str    
 11  Kms_Driven        111666 non-null  int64  
 12  Insurance_Valid   111666 non-null  int64  
 13  Service_History   111666 non-null  int64  
 14  Accidents         111666 non-null  int64  
 15  Tax_Paid          111666 non-null  int64  
 16  Number_of_Doors   111666 non-nu

In [8]:
car1_df.describe()

,Year,Mileage_kmpl,Engine_CC,Horsepower,Kms_Driven,Insurance_Valid,Service_History,Accidents,Tax_Paid,Number_of_Doors,Seats,Registration_Age,Price
count,111666.000000,102675.000000,102625.000000,102763.000000,1.116660e+05,111666.000000,111666.000000,111666.000000,111666.000000,111666.000000,111666.000000,111666.000000,1.116660e+05
mean,2011.998137,18.001595,1504.666422,120.338926,2.023779e+05,0.850895,0.752261,0.298372,0.901089,4.251348,5.160882,13.001863,1.009957e+06
std,7.216723,3.989785,386.478622,36.781208,1.746106e+05,0.356194,0.431701,0.544322,0.298544,0.697800,0.919267,7.216723,1.118583e+06
min,2000.000000,5.000000,800.000000,-18.509982,5.009000e+03,0.000000,0.000000,0.000000,0.000000,2.000000,2.000000,1.000000,5.000000e+04
25%,2006.000000,15.332062,1228.706761,94.703050,8.432350e+04,1.000000,1.000000,0.000000,1.000000,4.000000,5.000000,7.000000,3.422012e+05
50%,2012.000000,17.991263,1496.825645,119.747087,1.661400e+05,1.000000,1.000000,0.000000,1.000000,4.000000,5.000000,13.000000,7.585977e+05
75%,2018.000000,20.677573,1767.684665,145.194864,2.860740e+05,1.000000,1.000000,1.000000,1.000000,5.000000,5.000000,19.000000,1.292484e+06
max,2024.000000,34.760248,3245.326040,286.878228,4.344375e+06,1.000000,1.000000,5.000000,1.000000,5.000000,7.000000,25.000000,3.348553e+07


In [9]:
car1_df.head(10)

,Brand,Model,Year,Mileage_kmpl,Engine_CC,Horsepower,Fuel_Type,Transmission,Owner_Type,Color,City,Kms_Driven,Insurance_Valid,Service_History,Accidents,Tax_Paid,Number_of_Doors,Seats,Registration_Age,Price
0,Toyota,Camry,2001,19.615852,1396.560379,NaN,Hybrid,Automatic,First,Grey,Chennai,500928,1,1,0,1,4,5,24,3.531896e+05
1,Hyundai,i20,2012,19.478608,1130.771005,137.021719,Petrol,NaN,First,Grey,Mumbai,211510,0,1,0,1,4,5,13,9.596943e+05
2,Mahindra,Bolero,2008,17.469920,1766.466250,142.902961,NaN,Automatic,First,Black,Kolkata,333999,1,0,0,1,5,5,17,2.920229e+05
3,BMW,X1,2015,21.295421,NaN,49.782079,NaN,Manual,First,White,Pune,225930,1,1,3,1,4,5,10,2.949453e+06
4,Honda,Civic,2022,18.326833,1239.334935,121.505993,Diesel,Manual,Second,White,Kolkata,43404,1,0,0,1,5,5,3,6.365208e+05
5,Toyota,Camry,2007,NaN,1513.157430,100.927306,Petrol,Automatic,Second,NaN,NaN,321426,0,0,1,1,4,2,18,5.195264e+05
6,Hyundai,i20,2014,16.328654,1398.722600,111.846497,Diesel,Manual,First,NaN,Chennai,200343,1,0,0,1,4,4,11,4.953191e+05
7,Hyundai,Creta,2001,21.895120,800.000000,0.709766,Petrol,Manual,Second,White,Kolkata,228720,1,0,0,1,5,5,24,5.000000e+04
8,Volkswagen,Taigun,2020,17.461847,1397.891768,104.095168,Diesel,Automatic,Fourth+,Black,Hyderabad,122025,1,1,0,1,4,5,5,1.045347e+06
9,Audi,A4,2009,12.062221,1618.723936,86.071024,Petrol,Manual,Second,Blue,Pune,395200,1,1,1,1,5,7,16,2.633309e+05


In [10]:
car1_df.tail(10)

# YOU CAN ALSO USE iloc[] to peak through records at specific locations

,Brand,Model,Year,Mileage_kmpl,Engine_CC,Horsepower,Fuel_Type,Transmission,Owner_Type,Color,City,Kms_Driven,Insurance_Valid,Service_History,Accidents,Tax_Paid,Number_of_Doors,Seats,Registration_Age,Price
111656,Toyota,Camry,2024,NaN,1882.775963,140.219046,Petrol,Manual,First,Grey,Pune,24192,1,1,0,1,5,5,1,1.086233e+06
111657,Mahindra,XUV700,2003,19.817115,1395.617678,108.941422,Diesel,Manual,First,Brown,Bangalore,311014,0,1,0,1,4,4,22,5.000000e+04
111658,Mahindra,Scorpio,2012,25.428510,1088.664235,63.668722,Diesel,Manual,Second,NaN,NaN,187629,1,1,0,1,4,5,13,5.000000e+04
111659,Hyundai,Creta,2001,10.584043,1417.578170,142.016165,NaN,NaN,First,White,Hyderabad,312960,1,0,1,1,4,5,24,5.000000e+04
111660,BMW,X1,2019,18.229133,1505.604546,139.573791,Diesel,Manual,Third,NaN,Mumbai,133554,1,1,0,1,4,5,6,2.812521e+06
111661,Tata,Punch,2005,NaN,1761.529435,139.389685,Hybrid,Manual,Fourth+,Black,Kolkata,432240,1,1,0,1,5,5,20,5.000000e+04
111662,Tata,Harrier,2014,22.871347,1156.299967,85.496967,Petrol,Automatic,First,Blue,Kolkata,263604,1,1,0,1,4,5,11,4.894001e+05
111663,Ford,EcoSport,2008,23.291805,1336.926351,96.292228,Diesel,Manual,First,Blue,Pune,2423316,1,1,0,1,4,7,17,3.558083e+06
111664,Ford,Figo,2015,11.734531,2431.715340,NaN,Petrol,Manual,Second,Blue,NaN,229370,1,1,0,1,5,5,10,7.322810e+05
111665,Skoda,Octavia,2023,13.208398,1730.108663,125.115531,Petrol,Manual,First,Red,Chennai,44242,1,1,0,1,4,5,2,1.103364e+06
